<div style="text-align:center;">
  <h1>Generate DISP-S1 time series in Google Colab without handling python</h1>
  <h2>This tutorial demonstrates how to query and work with the OPERA DISP-S1 provisional data products from Earthdata repositories.</h2>
</div>

---    

### Data Used in the Example:   

- **Provisional OPERA Level 3 Surface Displacement (DISP-S1) Product from Sentinel-1 at 30m Resolution over North America**
    - The OPERA Level 3 Land-Surface Displacement (L3_DISP) product is generated from a sequence of the Level 2 Coregistered Single Look Complex (L2_CSLC) products derived from Sentinel-1AB (S1) satellites. The OPERA project aims to produce high-quality Interferometric Synthetic Aperture Radar (InSAR)-derived displacement data with reduced decorrelation noise using a hybrid Persistent Scatterer (PS) and Distributed Scatterer (DS) time series processing approach [[dolphin](https://joss.theoj.org/papers/10.21105/joss.06997)]. The displacement product provides information on anthropogenic and natural changes of Earth's surface, such as subsidence, tectonics, and landslides.
    
    - The OPERA project is generating geocoded DISP-S1 products over North America which includes USA and US Territories within 200 km from the US border, Canada, and all mainland countries from the southern US border down to and including Panama. Each pixel within a DISP-S1 frame contains geocoded displacement as well as quality layers information.

    - For more information about the OPERA project and other products please visit our website at https://www.jpl.nasa.gov/go/opera .

---

### OPERA DISP-S1 Data Architecture Overview

The OPERA DISP-S1 products are designed to simplify time series InSAR analysis for a wide user base. Key components of the DISP-S1 architecture:

- **Ministack Structure**  
  - Each file represents cumulative displacement between a fixed reference date and a specific secondary date
  - A **ministack** includes up to **15 acquisitions**: all files share the same reference date and have different secondary dates  
  - The **last file** in a ministack contains the **total cumulative displacement** from the reference date to the final (15th) secondary date

- **Sequential Ministacks**  
  - The **last secondary date of one ministack becomes the reference date** for the next ministack  
  - This creates a chain of overlapping stacks, where each new ministack starts at zero and spans the next 15 acquisitions

- **Spatial Referencing**  
  - Data is projected in the UTM zone appropriate to each frame  
  - Gridded at **30 m resolution**

- **Data Format & Layers**  
  - Provided as NetCDF files using CF-1.8 conventions
  - Contain:
    - Primary displacement data (`displacement`)
    - Quality metrics (`temporal_coherence`, `phase_similarity`, etc.)
    - Ancillary layers (`recommended_mask`, `water_mask`, `persistent_scatterer_mask`, etc.)
    - Spatial metadata (projection, pixel spacing, etc.)

- **Coverage & Frequency**  
  - Data available over North America, with a 6- to 12-day revisit depending on Sentinel-1 satellite orbit configuration  
  - Products are released as provisional, pending further validation

---

### OPERA DISP-S1 File Naming Convention

Each DISP-S1 product file follows a standardized naming pattern that encodes key metadata directly in the filename.

#### Example:
**OPERA_L3_DISP-S1_IW_F11116_VV_20240406T140838Z_20241003T140837Z_v1.1_20250220T140455Z.nc**

#### Field Descriptions:
- `OPERA_L3_DISP-S1` — OPERA Level-3 Displacement product from Sentinel-1
- `IW` — Interferometric Wide swath mode (Sentinel-1 acquisition mode)
- `<FRAMEID>` — Unique frame identifier (e.g., `F11116`)
- `<POL>` — Polarization used (`VV` or `VH`)
- `<REFDATE>` — Start date of the time series (reference date)
- `<SECONDDATE>` — End date of the time series (last acquisition in ministack)
- `<VERSION>` — Product version (e.g., `v1.1`)
- `<COMPUTEDATE>` — Date the product was generated

#### Notes:
- All displacement values in the file are **cumulative**, measured from `<REFDATE>` to `<SECONDDATE>`.
- Products with different `<REFDATE>` values represent separate ministacks.


Please refer to the [OPERA Product Specification Document](https://d2pn8kiwq2w21t.cloudfront.net/documents/OPERA_DISP_S1_Final_Product_Spec.pdf) for more details about the DISP-S1 products.

*Prepared by B. Raimbault using notebook contributions and python packages from M. Govorcin, and S. Staniewicz*
</div>



---



### ⚠️ Recommendation for Long-Term Time Series Analysis

For time series analysis spanning **longer than one year**, we recommend setting up a **local Python environment** on your laptop or institutional computing facilities.

<font color="red">⚠️ Google Colab runtimes are temporary — once disconnected or reset, all data and processing results are lost.</font>

Installing the processing environment locally ensures:
- Data persistence and reproducibility
- Better handling of large datasets over extended time ranges
- Greater control over computational resources

-- **Setting up your laptop conda environment:**

Assuming you have [miniconda](https://docs.conda.io/projects/miniconda/en/latest/miniconda-install.html) or conda installed. Open your terminal and run the following to open this file:
```
> conda create -n opera_disp-s1
> conda activate opera_disp-s1
> conda install -c conda-forge python==3.11.13 jupyter ipyleaflet gdal libgdal-netcdf
>  jupyter-notebook /path/to/your_file/Intro_to_DISP-S1_Colab.ipynb
```


If you are using this notebook on Google Colab, the python version must be 3.11.13, but if it's on your personnal machine no specific version is needed (but we do recommend python > 3.12)

---


# Environment Setup

Before working with DISP-S1 products, we need to install the necessary Python packages and tools used throughout this notebook in the Google Colab environment.

The following cell:
- Installs required dependencies
- Loads custom utility functions from the `displacement_tools` module

This setup ensures you have all components ready for visualizing and analyzing OPERA Level-3 displacement data.


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
!wget https://raw.githubusercontent.com/OPERA-Cal-Val/OPERA_Applications/main/DISP/Discover/setup_env.py -O setup_env.py
!wget https://raw.githubusercontent.com/OPERA-Cal-Val/OPERA_Applications/main/DISP/Discover/displacement_tools.py -O displacement_tools.py
from setup_env import install_dependencies
install_dependencies()
from displacement_tools import *




---



# Interactive Frame Explorer

Use the `display_opera_frames_map()` function below to visualize all available DISP-S1 frames over North America.

- 🟥 **Red frames** represent *ascending* Sentinel-1 tracks  
- 🟦 **Blue frames** represent *descending* Sentinel-1 tracks  

**Instructions:**
- Click on any frame to display:
  - The frame ID (4 to 5 digits)
  - The number of acquisition dates available for that frame
- Use the map controls or legend to toggle visibility between ascending and descending tracks

This interactive map is useful for identifying frame coverage and data availability for regions of interest (e.g., tectonics, subsidence, or earthquakes).

You can also go here: https://displacement.asf.alaska.edu to find the frame numbers associated to your area of interest and find interesting displacement features to look at.


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
display_opera_frames_map()



---



# Accessing Earthdata-Hosted Products

To download DISP-S1 products, you need to authenticate with NASA's Earthdata system, since the data is hosted on the Earthdata repository.

The following cell will prompt you to enter your **NASA Earthdata Login** credentials. This information is saved in a `.netrc` file within your Colab environment or laptop, enabling authenticated access for data downloads during this session.

<font color="red">After that, type your NASA Earthdata username, press Enter, then type your password and press Enter again.</font>  
<font color="red">If you make a mistake or if the following cells show download errors, rerun this cell and ensure the credentials are entered correctly.</font>

🔐 You can register for a free NASA Earthdata account here: [https://urs.earthdata.nasa.gov/](https://urs.earthdata.nasa.gov/)


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
#setup_earthdata_credentials()

### If on local machine, run the following line to set up the environment variables
netrc_file = "_netrc" if os.name == "nt" else ".netrc" ## default for Windows is _netrc, for Unix-like systems is .netrc
setup_earthdata_credentials(netrc_path=os.path.join(os.path.expanduser("~"), netrc_file))




---



### Selecting a DISP-S1 Frame and Time Range

In the next step, you'll be prompted to define the parameters for downloading a stack of DISP-S1 products.

You will be asked to enter:

1. **Frame ID** – Choose a frame number from the previous interactive map (click on a frame to get its ID).
2. **Start Date** – The beginning of the time series in the format `YYYY-MM-DD`.
3. **End Date** – The end of the time series in the format `YYYY-MM-DD`.

<font color="red">For testing purposes, we recommend using a short time window to keep the download and processing time fast and manageable.</font>

Once entered, the script will:
- Confirm the selected frame and time window
- Display how many DISP granules match your criteria
- Estimate the total download size

<font color="red">Make sure you have enough available space on your runtime. If the download size is too large, consider choosing a shorter time range or cropping later in the workflow.</font>

### **Science Dataset Example: San Simon / Bowie Subsidence in Arizona**
- We recommend looking at the San Simon / Bowie subsidence with a small data subset to try out this notebook (e.g., **Frame 34478, January 2022 to September 2024**).
- Here, we demonstrate how to create a time series of ground deformation for the San Simon / Bowie, AZ area using DISP-S1 products from January 2022 to September 2024. This region experiences noticeable subsidence primarily due to groundwater withdrawal. By analyzing the deformation over time, we can monitor the extent and progression of land sinking.



In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
products_df, frame_id, start_datetime, end_datetime, work_dir, nc_dir, nc_urls, username, password = prompt_user_and_search()



---



### Define Area of Interest (AOI) with a Bounding Box

Now that you've selected a DISP-S1 frame, you can interactively define a **bounding box** to focus your analysis on a specific region within the frame. You frame of interest is delineated in blue.

Use the map below to:

- Click the **black square button** ( ⬛ ) to activate the drawing tool  
- Then click on the map to draw a rectangular bounding box over your area of interest  
- Click on the map a second time to finish drawing  
- The selected bounds will be used to crop the displacement stack

This step helps reduce file size and speeds up processing by narrowing the data to your region of interest.

<font color="red">Tip: You can redraw the box by clicking the trash icon 🗑️, selecting <b>Clear All</b>, and starting over.</font>  
<font color="red">Note: Only the <b>last bounding box</b> drawn will be used for the analysis.</font>


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
bbox_bounds = setup_interactive_bbox_map(products_df)



---



# Estimate Cropped Stack Size

This step provides an approximate file size for the displacement stack based on the bounding box you drew earlier and the number of selected granules.

It helps you:
- Understand the size of the data you are about to download
- Decide whether to reduce the area of interest to save space

<font color="red">Warning: If the estimated size is too large for your available storage (~50GB), consider reducing the bounding box before continuing.</font>


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
file_size, subset_size, ratio, stack_gb, bbox = estimate_stack_size(
    nc_urls, bbox_bounds, username, password
)

print(f"Original file size: {file_size:.2f} MB")
print(f"Subset size: ~{subset_size:.2f} MB ({ratio:.2%} of original)")
print(f"Estimated total stack size: {stack_gb:.2f} GB")
print("⚠️  Ensure you have enough space or reduce the crop area.")



---



### Download Cropped DISP-S1 Data

Now that you've defined your area of interest and estimated the stack size, this step will download and crop the DISP-S1 data accordingly.

The `NUM_WORKERS` variable controls how many parallel download processes are used.  
<font color="red">⚠️ Note: Since available RAM in Google Colab is limited, it is mandatory to keep this number low (e.g., 2–3 workers).</font>

This process:
- Downloads each selected granule
- Crops them on-the-fly to match your bounding box
- Saves the output locally in your runtime environment

After a successful download, each granule's file path will be printed so you can verify that it completed correctly.


<font color="red"> It may take a few minutes depending on your settings and internet speed.</font>  
**Example:** Downloading and cropping 40 acquisitions with 3 workers typically takes around ~5 minutes, depending on the google colab / laptop internet speed.


<font color="red">🚨 If your notebook crashes or restarts unexpectedly, reduce the number of workers.</font>  
<font color="red"><b>IMPORTANT:</b> If Colab disconnects or crashes, you’ll need to re-run most previous cells (e.g., credentials, search, bounding box), but already downloaded files will remain available unless the runtime is fully reset.</font>

In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
NUM_WORKERS = 3
download_disp_files(nc_urls, bbox, nc_dir, username, password, NUM_WORKERS)



---



### (Optional) Extract Static Layers and Frame Metadata

This step extracts **static layers** (such as incidence angle, look angle, and line-of-sight information) and **metadata** associated with the DISP-S1 frame.  
These files are useful for advanced analysis or visualization, but not required for basic displacement time series workflows.

The static layers are saved in the `static` directory, and metadata is printed or stored for reference.

<font color="red">📌 This step is optional. Run it only if you need additional reference layers or frame details. It is a long run ~30 minutes at most.</font>


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
nc_files = list(nc_dir.glob("*.nc"))
meta = get_metadata(nc_files[0])
#static_layers.get_static_layers(nc_files[0], output_dir="static") # <--- Uncomment / Remove the # if you want to get the static layers.



---



### (Optional) Download and Plot DEM

This step downloads the **Digital Elevation Model (DEM)** corresponding to your selected DISP-S1 frame.  
The DEM is useful for visualizing topography and can be helpful for interpreting displacement signals affected by terrain.

- The DEM will be saved in the `static` directory
- If `show=True`, a preview of the DEM will be plotted inline

<font color="red">📌 This step is optional — run it if you want to visualize elevation or perform terrain-related analysis.</font>


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run

#download_and_plot_dem(nc_files[0], output_dir="static", show=True)  #<--- Uncomment / Remove the # if you want to get the dem information.



---



### Visualize Acquisition Dates and Ministack Structure

This step sets up the workspace and reads metadata for your selected DISP-S1 frame.  
It then displays a scatter plot of available acquisition pairs, based on the downloaded granules.

In DISP-S1:
- A **ministack** consists of up to 15 acquisition dates
- All displacements are measured relative to a fixed reference date
- The plot shows:
  - **x-axis**: reference date (the same across the stack)
  - **y-axis**: each secondary date paired with that reference
- This results in a vertical alignment of dots in the scatter plot: one column per ministack

Each point represents a pair of dates over which cumulative displacement is measured from the reference.


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
work_dir, nc_dir, nc_files, meta = setup_workspace_and_metadata(frame_id)
disp_df, versions = get_disp_versions(nc_dir)
plot_date_scatter_by_version(versions)



---



### Explore the Contents of the DISP-S1 Data Stack

The `stack_prod` object is an `xarray.Dataset` that represents the full ministack of DISP-S1 data loaded from the downloaded DISP-S1 NetCDF files.

As shown below, the dataset includes:
- **Coordinates**: spatial (x, y) and temporal (time) axes
- **Data variables**: multiple layers beyond just displacement
- **Attributes**: metadata describing the product version, projection, mission, and contact info

While the **`displacement`** variable is the main product of interest (cumulative surface motion in the LOS from the reference date), DISP-S1 products also include a variety of supporting layers.

<font color="red">📌 A more detailed explanation of each layer will be provided later in the notebook.</font>


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
stack_prod, epsg = build_stack_and_get_epsg(disp_df)
stack_prod



---



### Select a Reference Point for Displacement

InSAR displacement measurements are **relative** in both space and time — they show movement **with respect to a chosen reference point**.  
To interpret the displacement meaningfully, we need to define a point that is assumed to be:
- Not moving during the observed time period  
- Or located near your area of interest or object, to capture differential motion relative to it

In this step:
- Click **"Enable Point Click"** to turn on the interactive map
- Click on the map to select your reference point
- The **last point clicked** will be used to spatially reference your entire displacement stack

<font color="red">The reference point will serve as the zero-displacement baseline, and all other pixel values in the stack will be computed relative to it.
It means that if this points is noisy or unreliable, every other pixels will be affected.</font>

What you see on the map is the displacement value for the **last date in the stack**, meaning it shows the **total cumulative deformation** over the full time period.
- Colors represent motion in the satellite's line-of-sight (LOS) direction  
  - **Red** indicates motion **away** from the satellite  
  - **Blue** indicates motion **toward** the satellite

<font color="red">📌 Only the final point clicked will be used — click again if you need to change it.</font>  
<font color="red"> Choosing a good reference is important: a stable or nearby point helps isolate the motion you're investigating.</font>


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
clicked_refe = create_reference_selection_map(stack_prod, meta)



---



### Visualize Cumulative Displacement Relative to Reference Point

In this step, we finalize the spatial referencing by applying the selected reference point to the displacement stack.

The map shows the cumulative displacement from the reference date to the last acquisition date in the stack:
- Colors represent motion in the satellite's line-of-sight (LOS) direction  
  - **Red** indicates motion **away** from the satellite  
  - **Blue** indicates motion **toward** the satellite
- The green dot marks the **reference point**, which has been set to zero displacement
- The values are in **centimeters**, relative to that reference

<font color="red">📌 Reminder: The accuracy of this map depends on the quality and appropriateness of your selected reference point.</font>


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
stack_prod = set_reference_point(stack_prod, clicked_refe[-1])
plot_displacement_map(stack_prod, meta, clicked_refe[-1], buffer_fraction=0.05)



---



### Interactive Displacement Explorer

This interactive map brings together several key tools for exploring the displacement stack:

1. **Plot Cumulative Displacement**  
   By default, the map displays the displacement at the last date in the time series, relative to the current reference point.

2. **Select or Update Reference Point**  
   - Click **"Enable Reference Click"** to choose a new reference point on the map  
   - The map will automatically update using this new point as the zero-displacement baseline  
   - Useful for re-centering around different stable or local-interest locations  
   - A **red dot** will appear on the map to mark the selected reference point

3. **Plot Time Series at a Pixel**  
   - Click **"Enable Plot Click"**, then click anywhere on the map  
   - Then click **"Plot Time Series"** to visualize the cumulative displacement over time at the selected point  
   - This is helpful for analyzing the motion history of specific locations (e.g., faults, infrastructure, or subsiding areas)

   <font color="red">📌 If you select a new pixel, you must click <b>"Plot Time Series"</b> again to refresh the graph.</font>

4. **Time Slider**  
   Use the time slider to preview displacement at each acquisition date across the stack.  
   This lets you observe the evolution of surface deformation over time.

<font color="red">📌 All interactions are linked: changing the reference point resets the displacement baseline across the entire stack and updates the map accordingly. You must click <b>"Plot Time Series"</b> again to refresh the time series scatter plot.</font>  
<font color="red">⏳ Please give a few seconds between actions, as the interactive plotter may respond slowly (depending on the stack size). After choosing a reference point, the map will update and show a red dot indicating the location you selected.</font>


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
setup_interactive_viewer(stack_prod)

---

### Velocity Estimation

This cell computes a linear velocity map (in cm/year) from the displacement time series using least squares fitting.
- Fits a linear model: *displacement = velocity × time + intercept*.
- Colors represent motion in the satellite's line-of-sight (LOS) direction  
  - **Red** indicates motion **away** from the satellite  
  - **Blue** indicates motion **toward** the satellite
- The green dot marks the **reference point**, which has been set to zero displacement

In [ ]:
rate = compute_velocity_from_stack(stack_prod, displacement_var="displacement")
plot_displacement_map(stack_prod, meta, clicked_refe[-1], buffer_fraction=0.05, velocity=rate)



---



### Inspect DISP-S1 Data Layers and Their Attributes

This step prints detailed information about the variables contained in the DISP-S1 data stack.

For each variable (e.g., displacement, coherence, phase similarity), the function will display:
- The data type, and description of what the layer represents
- Units, if available
- Any relevant metadata or processing notes



In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
stack_quality, ministack_ref = extract_ministack_references(stack_prod)
print_stackprod_variable_info(stack_prod)


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
ministack_ref.persistent_scatterer_mask



---



### Quality Metrics

This step provides a summary of the **spatial quality** of the ministack by computing percentages of pixel characteristics:

- **% Persistent Scatterers (PS)**  
  The fraction of pixels identified as persistent scatterers, i.e., stable points that consistently reflect radar signals and are reliable for long-term displacement monitoring.

- **% Valid Time Series Pixels (recommended_mask)**  
  The proportion of pixels flagged as “good” by the recommended quality mask, combining multiple filters such as coherence, water masking, and phase similarity.

- **% Connected Component Valid (not 0)**  
  Indicates the portion of pixels that are part of a valid, unwrapped phase region (i.e., not isolated or ambiguous).

These percentages provide a diagnostic to assess the **spatial coverage and reliability** of the stack.

<font color="red">📌 High values suggest that the stack is well-suited for time series analysis. Low values may indicate decorrelation or poor phase quality in the region.</font>


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
plot_quality_summary(ministack_ref, stack_quality)




---



### Quality Metrics

This step computes and displays quality indicators for the displacement stack based on metrics derived from the data:

- **Median Temporal Coherence**: Indicates the overall stability of the interferometric signal over time. Higher values suggest more reliable measurements.

- **Median Phase Similarity**: Reflects how consistent the phase is within local neighborhoods. Useful for identifying noisy or unstable areas.

- **Total Number of 2π Phase Jumps**: A diagnostic for phase unwrapping issues. A high count may indicate discontinuities or noise artifacts.

These metrics provide a summary of the stack’s integrity and can help you decide whether to proceed with or filter out unreliable areas.


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
plot_advanced_quality_metrics(stack_quality)




---



### SHP (Statistically Homogeneous Pixels) Statistics

This step computes and plots statistics related to **`shp_counts`**, which represent the number of statistically homogeneous pixels (SHPs) used in the multilooking process at each pixel.

For the selected ministack, this function reports:
- The **median** number of SHPs per pixel
- The **standard deviation** of SHP counts across the scene

These metrics help assess the robustness of the distributed scatterer (DS) processing:
- A higher median indicates more reliable local statistics
- High variability (standard deviation) may point to inconsistent signal quality across the scene

In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
plot_shp_stats(ministack_ref)




---


### Export DISP-S1 Stack for External Use (MintPy)

This final step prepares your displacement time series for export and use in other tools such as **MintPy**.

The process involves two main actions:

1. **Reformat DISP-S1 Files into a Single NetCDF Stack**  
   - Combines multiple DISP-S1 granules (from a ministack) into a single NetCDF file  
   - Uses a defined reference method (`NONE / MEDIAN / BORDER / POINT /  HIGH COHERENCE`) to re-reference all dates consistently  
   - Drops unnecessary variables like `connected_component_labels`, `shp_counts`, etc. to reduce file size  
   - Output: a single file named `disp-output-<frame_id>.nc`.

2. **Convert to MintPy Format**  
   - Uses the exported NetCDF to generate a MintPy-compatible displacement time series  
   - Produces outputs in the `export/` directory  

In [ ]:
# -----------------------------
# DISP-S1 Stack Reformat Script
# -----------------------------

# Set the reference method to one of the following:
#   "NONE"            → No referencing (raw displacement)
#   "POINT"           → Reference to a specific lat/lon location
#   "MEDIAN"          → Median over valid land pixels (excludes water)
#   "BORDER"          → Median over border pixels (configurable size)
#   "HIGH_COHERENCE"  → Median over high-coherence mask (based on threshold)
REFERENCE_METHOD = "BORDER"  # <- Change this line to use a different method

# Output name based on frame ID
OUTPUT_NAME = f"disp-output-{frame_id}.nc"

# Input files
INPUT_FILES = sorted(glob.glob(f"{frame_id}/OPERA_DISP_S1_Files/*.nc"))
input_files_str = " ".join(INPUT_FILES)

# Execute the reformat command
!opera-utils disp-s1-reformat \
    --drop-vars connected_component_labels shp_counts persistent_scatterer_mask timeseries_inversion_residuals short_wavelength_displacement \
    --reference-method {REFERENCE_METHOD} \
    --output-name {OUTPUT_NAME} \
    --input-files {input_files_str}
    # Additional reference parameters (uncomment as needed):
    # --reference-lat 34.123 --reference-lon -117.456          # For POINT method
    # --reference-border-pixels 3                              # For BORDER method (default is 3)
    # --reference-coherence-threshold 0.7                      # For HIGH_COHERENCE method (default is 0.7)


# Convert to MintPy
SAMPLE_FILE = INPUT_FILES[0]
!python -m opera_utils.disp.mintpy {OUTPUT_NAME} \
    --sample-disp-nc {SAMPLE_FILE} \
    --outdir export

!mv {OUTPUT_NAME} export/



---

### Export for GIS & Quick-Look

The following code cell exports your DISP-S1 time series in two formats for further analysis or visualization:

- **Multi-band GeoTIFF Export**  
  A single GeoTIFF is created with one band per acquisition date. Each band contains cumulative displacement from the reference date to the corresponding acquisition.  
  This file can be opened in GIS software such as QGIS or ArcGIS Pro.

- **PNG Snapshots**  
  One PNG is generated per acquisition date to provide a quick visual overview of displacement evolution.  
  All images use a consistent, symmetric color scale centered on zero, determined from the final (most cumulative) image.

In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
export_timeseries_to_geotiff(f"export/disp-output-{frame_id}.nc", frame_id=frame_id)
export_timeseries_pngs(f"export/disp-output-{frame_id}.nc", frame_id=frame_id)

### Download Results from Google Colab

After exporting your processed displacement stack and MintPy-compatible files, you can download them directly from the Colab environment to your local machine.

There are two common methods:

---

#### Option 1: Use the File Browser (GUI)
1. On the left side of the Colab window, click the folder icon 📁
2. Navigate to the directory containing your files (e.g., `export/` or the `.nc` files in `frame_id/OPERA_DISP_S1_Files/*.nc`)
3. Right-click on a file and select **“Download”**

---

#### Option 2: Use Python Code
Use the following snippet to trigger a file download via code:

```python
from google.colab import files
!zip -r mintpy_outputs.zip export/
files.download("mintpy_outputs.zip")
```
Note that you can load any OPERA DISP-S1 NETCDF file in QGIS (using drag-and-drop) and select your layers of interest in the popup window.


In [ ]:
# This is a code cell — click to select, then press Shift+Enter or click the ▶️ play button on the left side of the cell in Google Colab to run
from google.colab import files
!rm -rf mintpy_outputs.zip
!zip -r mintpy_outputs.zip export/
files.download("mintpy_outputs.zip")

# EOF